# Evaluate similarity suggestions

In [1]:
%load_ext autoreload

In [2]:
import gc
import os
import pickle
import warnings
from os.path import join

import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display, Markdown, display_html

In [3]:
%autoreload
from datasim.dataset_ot import DatasetMapping

## Load and preprocess data

In [4]:
DATA_PATH = "/vol/data/dataset-similarity/preprocessed"
CACHE_DIR = "/vol/data/dataset-similarity/cache"
FIG_DIR = "/vol/data/dataset-similarity/figures"


QUERY_DATASET = "7d7cabfd-1d1f-40af-96b7-26a0825a306d"
REF_DATASET = "ced320a1-29f3-47c1-a735-513c7084d508"
N_TOP_GENES = 3000

In [5]:
cache_file = join(
    CACHE_DIR, 
    "+".join([QUERY_DATASET, REF_DATASET, f"{N_TOP_GENES}HVGs"]) + ".pickle"
)
if os.path.isfile(cache_file):
    print("Using cached files...")
    with open(cache_file, "rb") as f:
        adata_query, adata_ref = pickle.load(f)
else:
    # load and preprocess data
    adata_query, adata_ref = DatasetMapping.preprocess_adatas(
        sc.read_h5ad(join(DATA_PATH, f"{QUERY_DATASET}.h5ad")),
        sc.read_h5ad(join(DATA_PATH, f"{REF_DATASET}.h5ad")),
        n_top_genes=N_TOP_GENES
    )
    with open(cache_file, "wb") as f:
        pickle.dump((adata_query, adata_ref), f)

adata_query.obs["cell_type_author"] = adata_query.obs["ct2"]
gc.collect();

Using cached files...


In [6]:
# normalize data for DE tests
sc.pp.normalize_total(adata_query, target_sum=1e4)
sc.pp.normalize_total(adata_ref, target_sum=1e4)
sc.pp.log1p(adata_query)
sc.pp.log1p(adata_ref)

In [7]:
# Use human readable gene names for evaluation
adata_query.var.set_index("feature_name", inplace=True)
adata_ref.var.set_index("feature_name", inplace=True)

In [8]:
adata_query

AnnData object with n_obs × n_vars = 600929 × 3000
    obs: 'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 'is_primary_data', 'sex', 'suspension_type', 'tissue', 'ct1', 'ct2', 'ct3', 'cell_type_author'
    uns: 'log1p'

In [9]:
adata_ref

AnnData object with n_obs × n_vars = 1058909 × 3000
    obs: 'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 'is_primary_data', 'sex', 'suspension_type', 'tissue', 'author_cell_type', 'cell_type_author'
    uns: 'log1p'

In [10]:
MODEL_VERSION = "_n_genes_de_gene_overlap=10+tau=1.00+n_top_genes=3000"

cluster_mapping = pd.read_parquet(f"/vol/data/dataset-similarity/model-output/cluster_mapping{MODEL_VERSION}.parquet")
cluster_distance = pd.read_parquet(f"/vol/data/dataset-similarity/model-output/cluster_distance{MODEL_VERSION}.parquet")

In [11]:
def extract_ontology_mapping(adata):
    return (
        adata.obs[["cell_type_author", "cell_type"]]
        .drop_duplicates()
        .set_index("cell_type_author")["cell_type"]
        .to_dict()
    )


ontology_mapping_query = extract_ontology_mapping(adata_query)
ontology_mapping_ref = extract_ontology_mapping(adata_ref)

## Select most similar clusters

In [12]:
top_n_labels = DatasetMapping.select_most_similar_clusters(
    cluster_mapping, 
    cluster_distance, 
    threshold_mass=0.25,
    threshold_distance=1.1, 
    n_top=None
)
top_n_labels

{'B_Mem': ['IGHMhi_memory_B'],
 'B_Mem_Prolif': ['IGHMlo_memory_B'],
 'B_Naive': ['naive_B'],
 'B_Preplasma': ['atypical_B'],
 'NKT': ['CD4+_T_cyt', 'NK'],
 'NK_CD16+': ['CD16+_NK'],
 'NK_CD56++': ['CD56+_NK'],
 'NK_Prolif': [],
 'PB': ['Plasma_B'],
 'PB_Prolif': [],
 'Progen_CLP': [],
 'Progen_CMP': [],
 'Progen_MEP': [],
 'Progen_MPP': [],
 'T4_Mem': ['CD4+_T_cm'],
 'T4_Mem_Prolif': [],
 'T4_Naive': ['CD4+_T_naive'],
 'T4_Treg': ['Treg'],
 'T8_MAIT': ['MAIT'],
 'T8_Mem': ['CD8+_T_GZMB+'],
 'T8_Mem_Prolif': ['CD8+_T_GZMK+'],
 'T8_Naive': ['CD8+_T_naive'],
 'T_NK_Prolif': [],
 'Tgd_1': ['CD4+_T_cyt'],
 'Tgd_2': ['gdT'],
 'cDC_1': ['cDC1'],
 'cDC_2': ['cDC2', 'cDC'],
 'cM': ['CD14+_Monocyte'],
 'ncM': ['CD16+_Monocyte'],
 'pDC': ['pDC']}

#### Author provided cluster labels

In [13]:
for i, (k, v) in enumerate(top_n_labels.items()):
    display(Markdown(f"*{i+1}*: **{k}**: {v}"))

*1*: **B_Mem**: ['IGHMhi_memory_B']

*2*: **B_Mem_Prolif**: ['IGHMlo_memory_B']

*3*: **B_Naive**: ['naive_B']

*4*: **B_Preplasma**: ['atypical_B']

*5*: **NKT**: ['CD4+_T_cyt', 'NK']

...

*26*: **cDC_1**: ['cDC1']

*27*: **cDC_2**: ['cDC2', 'cDC']

*28*: **cM**: ['CD14+_Monocyte']

*29*: **ncM**: ['CD16+_Monocyte']

*30*: **pDC**: ['pDC']

#### Ontology mapped cluster labels

In [14]:
for i, (k, v) in enumerate(top_n_labels.items()):
    display(Markdown(f"*{i+1}*: **{ontology_mapping_query[k]}**: {[ontology_mapping_ref[elem] for elem in v]}"))

*1*: **B cell**: ['memory B cell']

*2*: **B cell**: ['memory B cell']

*3*: **B cell**: ['naive B cell']

*4*: **B cell**: ['mature B cell']

*5*: **natural killer cell**: ['CD4-positive, alpha-beta cytotoxic T cell', 'natural killer cell']

...

*26*: **conventional dendritic cell**: ['CD141-positive myeloid dendritic cell']

*27*: **conventional dendritic cell**: ['CD1c-positive myeloid dendritic cell', 'conventional dendritic cell']

*28*: **classical monocyte**: ['CD14-positive monocyte']

*29*: **non-classical monocyte**: ['CD14-low, CD16-positive monocyte']

*30*: **plasmacytoid dendritic cell**: ['plasmacytoid dendritic cell']

## Evaluate cluster similarity

In [15]:
%autoreload
from datasim.utils import get_differentially_expressed_genes

In [16]:
def highly_expressed_genes(adata, n_genes):
    highly_expressed = {}
    
    for cluster in adata.obs["cell_type_author"].unique():
        avg_expression = np.array(
            adata[adata.obs["cell_type_author"] == cluster].X.mean(axis=0)
        ).flatten()
        highly_expressed_genes_idxs = np.argsort(-avg_expression)[:n_genes]
        highly_expressed[cluster] = {
            "gene": adata.var.index[highly_expressed_genes_idxs].tolist(),
            "average_expression": avg_expression[highly_expressed_genes_idxs]
        }

    return highly_expressed


In [17]:
METHOD = "wilcoxon"
P_VAL_THRESHOLD = 0.01
N_TOP_GENES = 10
N_TOP_GENES_BORDER = 10

# ignore warnings here as scanpy.tl.rank_genes_groups throws a lot of warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    de_genes_query = get_differentially_expressed_genes(
        adata_query, 
        "cell_type_author", 
        n_genes=N_TOP_GENES + N_TOP_GENES_BORDER,
        method=METHOD,
        p_value_threshold=P_VAL_THRESHOLD
    )
    de_genes_ref = get_differentially_expressed_genes(
        adata_ref, 
        "cell_type_author", 
        n_genes=N_TOP_GENES + N_TOP_GENES_BORDER,
        method=METHOD,
        p_value_threshold=P_VAL_THRESHOLD
    )


In [18]:
highly_expressed_query = highly_expressed_genes(
    adata_query, n_genes=N_TOP_GENES + N_TOP_GENES_BORDER
)
highly_expressed_ref = highly_expressed_genes(
    adata_ref, n_genes=N_TOP_GENES + N_TOP_GENES_BORDER
)

In [19]:
def to_hex(m, val):
    rgba = m.to_rgba(val)
    r, g, b, _ = rgba
    return "#{:02x}{:02x}{:02x}".format(int(r*255), int(g*255), int(b*255))


def style_map_val(v, props=""):
    m = cm.ScalarMappable(
        norm=mpl.colors.Normalize(vmin=0.0, vmax=1.0), 
        cmap=plt.get_cmap("Greens")
    )
    return f"background:{to_hex(m, v)};"


def style_dist_val(v, props=""):
    m = cm.ScalarMappable(
        norm=mpl.colors.Normalize(vmin=0.0, vmax=2.0), 
        cmap=plt.get_cmap("RdYlGn_r")
    )
    return f"background:{to_hex(m, v)};"


In [20]:
def create_two_column_layout(left_column_html, right_column_html):
    template = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Two Columns Layout</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 0;
                padding: 0;
            }}

            .container {{
                display: flex;
            }}

            .column {{
                flex: 1; /* Each column takes up an equal amount of space */
                padding: 5px; /* Optional: adds some padding inside the columns */
            }}

            .left-column {{
                flex: 55;
                background-color: #f1f1f1; /* Optional: background color for left column */
            }}

            .right-column {{
                flex: 45;
                background-color: #e2e2e2; /* Optional: background color for right column */
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="column left-column">
                {left_column_html}
            </div>
            <div class="column right-column">
                {right_column_html}
            </div>
        </div>
    </body>
    </html>
    """

    return template


In [21]:
for k, v in top_n_labels.items():
    mapping = cluster_mapping.loc[k]
    distance = cluster_distance.loc[k]
    html = [f"<h1> <b>{k}</b>: </h1>"]

    if v:
        # Add summary for suggestions
        html_suggestions = []
        ct_name, map_vals, dist_vals = [], [], []
        for s in v:
            ct_name.append(s)
            map_vals.append(mapping[s])
            dist_vals.append(distance[s])
        html_suggestions.append(
            pd.DataFrame({"OT mass": map_vals, "distance": dist_vals}, index=ct_name)
            .style
            .format("{:.2f}")
            .map(style_map_val, subset=["OT mass"])
            .map(style_dist_val, subset=["distance"])
            .set_table_attributes("style='display:inline'")
            ._repr_html_()
        )
        html_suggestions.append("<br /><br />")
        # Add summary for differentially expressed genes
        html_de_overlap = ["<h4>Overlap of DE genes</h4>"]

        def style_overlap(v, props=""):
            top_n_genes = de_genes_query[k]["gene"].iloc[:N_TOP_GENES].tolist()
            top_n_genes_border = de_genes_query[k]["gene"].iloc[
                N_TOP_GENES:N_TOP_GENES+N_TOP_GENES_BORDER
            ].tolist()
            if v in top_n_genes:
                return "color:green;"
            elif v in top_n_genes_border:
                return "color:orange;"
            else:
                return "color:red;"

        html_de_overlap += [
            de_genes_query[k]
            .head(N_TOP_GENES)
            .copy()
            .style
            .format("{:.2f}", subset=["logfoldchg", "score"])
            .format("{:.3f}", subset=["pval_adj"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"QUERY - {k}")
            ._repr_html_()
        ]
        html_de_overlap += [
            de_genes_ref[gene]
            .head(N_TOP_GENES)
            .copy()
            .style
            .format("{:.2f}", subset=["logfoldchg", "score"])
            .format("{:.3f}", subset=["pval_adj"])
            .map(style_overlap, subset=["gene"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"REF - {gene}")
            ._repr_html_()
            for gene in v
        ]
        html_de_overlap.append("<br /><br />")

        # Add summary for highly expressed genes
        html_he_overlap = ["<h4>Overlap of highly expressed genes</h4>"]

        def style_overlap(v, propgs=""):
            top_n_genes = highly_expressed_query[k]["gene"][:N_TOP_GENES]
            top_n_genes_border = highly_expressed_query[k]["gene"][
                N_TOP_GENES:N_TOP_GENES+N_TOP_GENES_BORDER
            ]
            if v in top_n_genes:
                return "color:green;"
            elif v in top_n_genes_border:
                return "color:orange;"
            else:
                return "color:red;"

        html_he_overlap += [
            pd.DataFrame(highly_expressed_query[k])
            .head(N_TOP_GENES)
            .style
            .format("{:.2f}", subset=["average_expression"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"QUERY - {k}")
            ._repr_html_()
        ]
        html_he_overlap += [
            pd.DataFrame(highly_expressed_ref[gene])
            .head(N_TOP_GENES)
            .style
            .format("{:.2f}", subset=["average_expression"])
            .map(style_overlap, subset=["gene"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"REF - {gene}")
            ._repr_html_()
            for gene in v
        ]
    else:
        html_de_overlap = []
        html_he_overlap = []

    if html_de_overlap and html_he_overlap:
        html = html + html_suggestions + [create_two_column_layout("".join(html_de_overlap), "".join(html_he_overlap))]
    else:
        html = html + ["<b><i>No matches found</i></b>"]
    display_html("".join(html), raw=True)

    with open(join("cluster-evaluation-output", f"{k}.html"), "w") as f:
        f.write("".join(html))

    print("\n")


B_Mem : 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 IGHMhi_memory_B 
 0.94 
 0.76 
 
 
 
 
 <!DOCTYPE html>
 
 
 
 
 Two Columns Layout 
 
 
 
 
 
 Overlap of DE genes 
 
 QUERY - B_Mem 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 CD79A 
 6.39 
 0.000 
 124.47 
 
 
 1 
 MS4A1 
 6.21 
 0.000 
 120.85 
 
 
 2 
 NXPH4 
 5.66 
 0.000 
 10.47 
 
 
 3 
 LINC01857 
 5.48 
 0.000 
 52.27 
 
 
 4 
 BANK1 
 5.18 
 0.000 
 93.17 
 
 
 5 
 LINC01781 
 5.14 
 0.000 
 38.52 
 
 
 6 
 TNFRSF13B 
 4.96 
 0.000 
 40.70 
 
 
 7 
 FCRL2 
 4.90 
 0.000 
 33.99 
 
 
 8 
 RALGPS2 
 4.86 
 0.000 
 88.65 
 
 
 9 
 HLA-DRA 
 4.81 
 0.000 
 118.48 
 
 
 

 
 REF - IGHMhi_memory_B 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 MS4A1 
 7.45 
 0.000 
 179.23 
 
 
 1 
 CD79A 
 7.43 
 0.000 
 180.24 
 
 
 2 
 ALPL 
 7.29 
 0.000 
 8.86 
 
 
 3 
 BANK1 
 6.69 
 0.000 
 176.04 
 
 
 4 
 LINC01857 
 6.69 
 0.000 
 106.14 
 
 
 5 
 GALNTL6 
 6.36 
 0.000 
 9.96 
 
 
 6 
 TNFRSF13B 
 6.25 
 0.000 
 109.28 
 
 
 7 
 FCRL2 
 5.99 
 0.000 
 102.03 
 
 
 8 
 SEMA3D 
 5.98 
 0.000 
 4.44 
 
 
 9 
 RALGPS2 
 5.97 
 0.000 
 170.00 
 
 
 
 
 
 
 Overlap of highly expressed genes 
 
 QUERY - B_Mem 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 MALAT1 
 6.70 
 
 
 1 
 CD74 
 6.41 
 
 
 2 
 EEF1A1 
 6.40 
 
 
 3 
 RPS12 
 6.03 
 
 
 4 
 B2M 
 5.93 
 
 
 5 
 MT-CO1 
 5.73 
 
 
 6 
 RPL39 
 5.70 
 
 
 7 
 RPS6 
 5.19 
 
 
 8 
 RPS4X 
 5.13 
 
 
 9 
 HLA-DRA 
 5.11 
 
 
 

 
 REF - IGHMhi_memory_B 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 MALAT1 
 6.69 
 
 
 1 
 CD74 
 6.47 
 
 
 2 
 EEF1A1 
 5.89 
 
 
 3 
 MT-CO1 
 5.81 
 
 
 4 
 RPL41 
 5.80 
 
 
 5 
 RPS12 
 5.68 
 
 
 6 
 B2M 
 5.65 
 
 
 7 
 RPS6 
 5.14 
 
 
 8 
 RPS4X 
 5.09 
 
 
 9 
 HLA-DRA 
 5.07

B_Mem_Prolif : 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 IGHMlo_memory_B 
 1.00 
 0.62 
 
 
 
 
 <!DOCTYPE html>
 
 
 
 
 Two Columns Layout 
 
 
 
 
 
 Overlap of DE genes 
 
 QUERY - B_Mem_Prolif 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 COCH 
 7.53 
 0.000 
 45.25 
 
 
 1 
 CHAD 
 6.92 
 0.000 
 8.93 
 
 
 2 
 SSPN 
 6.91 
 0.000 
 37.32 
 
 
 3 
 LINC01781 
 6.65 
 0.000 
 69.55 
 
 
 4 
 TEX9 
 6.60 
 0.000 
 31.14 
 
 
 5 
 MS4A1 
 6.45 
 0.000 
 130.12 
 
 
 6 
 CD79A 
 6.29 
 0.000 
 127.80 
 
 
 7 
 IGHE 
 6.20 
 0.000 
 14.27 
 
 
 8 
 GRAMD1C 
 5.98 
 0.000 
 35.25 
 
 
 9 
 AIM2 
 5.68 
 0.000 
 62.40 
 
 
 

 
 REF - IGHMlo_memory_B 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 COCH 
 8.01 
 0.000 
 95.69 
 
 
 1 
 SSPN 
 7.53 
 0.000 
 80.33 
 
 
 2 
 LINC01781 
 7.47 
 0.000 
 117.37 
 
 
 3 
 MS4A1 
 7.35 
 0.000 
 214.02 
 
 
 4 
 CHAD 
 7.27 
 0.000 
 22.52 
 
 
 5 
 CD79A 
 7.14 
 0.000 
 212.46 
 
 
 6 
 BANK1 
 6.73 
 0.000 
 211.25 
 
 
 7 
 TEX9 
 6.50 
 0.000 
 61.67 
 
 
 8 
 IGHE 
 6.37 
 0.000 
 17.99 
 
 
 9 
 TNFRSF13B 
 6.35 
 0.000 
 124.15 
 
 
 
 
 
 
 Overlap of highly expressed genes 
 
 QUERY - B_Mem_Prolif 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 MALAT1 
 6.57 
 
 
 1 
 CD74 
 6.48 
 
 
 2 
 EEF1A1 
 6.39 
 
 
 3 
 B2M 
 6.00 
 
 
 4 
 RPS12 
 5.95 
 
 
 5 
 RPL39 
 5.63 
 
 
 6 
 MT-CO1 
 5.59 
 
 
 7 
 ACTB 
 5.34 
 
 
 8 
 TMSB4X 
 5.32 
 
 
 9 
 HLA-DRA 
 5.26 
 
 
 

 
 REF - IGHMlo_memory_B 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 MALAT1 
 6.62 
 
 
 1 
 CD74 
 6.54 
 
 
 2 
 EEF1A1 
 5.98 
 
 
 3 
 RPL41 
 5.85 
 
 
 4 
 MT-CO1 
 5.84 
 
 
 5 
 RPS12 
 5.79 
 
 
 6 
 B2M 
 5.72 
 
 
 7 
 HLA-DRA 
 5.17 
 
 
 8 
 RPS6 
 5.16 
 
 
 9 
 RPS4X 
 5.14

B_Naive : 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 naive_B 
 1.00 
 0.63 
 
 
 
 
 <!DOCTYPE html>
 
 
 
 
 Two Columns Layout 
 
 
 
 
 
 Overlap of DE genes 
 
 QUERY - B_Naive 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 TCL1A 
 8.98 
 0.000 
 238.58 
 
 
 1 
 RP11-164H13.1 
 8.27 
 0.000 
 34.48 
 
 
 2 
 RP11-265P11.1 
 7.96 
 0.000 
 8.20 
 
 
 3 
 CD79A 
 7.61 
 0.000 
 308.48 
 
 
 4 
 TCL1B 
 7.42 
 0.000 
 8.91 
 
 
 5 
 SYN3 
 7.38 
 0.006 
 2.95 
 
 
 6 
 MS4A1 
 7.33 
 0.000 
 294.22 
 
 
 7 
 KCNG1 
 7.26 
 0.000 
 10.53 
 
 
 8 
 FCER2 
 7.24 
 0.000 
 191.84 
 
 
 9 
 SLC38A11 
 6.91 
 0.000 
 14.30 
 
 
 

 
 REF - naive_B 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 TCL1A 
 10.35 
 0.000 
 279.90 
 
 
 1 
 RP11-164H13.1 
 8.95 
 0.000 
 66.04 
 
 
 2 
 RP11-265P11.1 
 8.85 
 0.000 
 30.20 
 
 
 3 
 FCER2 
 8.15 
 0.000 
 268.12 
 
 
 4 
 KCNG1 
 8.11 
 0.000 
 63.24 
 
 
 5 
 CD79A 
 8.03 
 0.000 
 297.51 
 
 
 6 
 MS4A1 
 8.00 
 0.000 
 293.99 
 
 
 7 
 NKX6-3 
 7.92 
 0.000 
 4.65 
 
 
 8 
 TCL1B 
 7.92 
 0.000 
 10.29 
 
 
 9 
 IGHM 
 7.86 
 0.000 
 287.83 
 
 
 
 
 
 
 Overlap of highly expressed genes 
 
 QUERY - B_Naive 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 6.62 
 
 
 1 
 MALAT1 
 6.53 
 
 
 2 
 EEF1A1 
 6.18 
 
 
 3 
 RPS12 
 5.84 
 
 
 4 
 B2M 
 5.69 
 
 
 5 
 RPL39 
 5.58 
 
 
 6 
 MT-CO1 
 5.58 
 
 
 7 
 HLA-DRA 
 5.30 
 
 
 8 
 TMSB4X 
 5.12 
 
 
 9 
 RPS4X 
 4.97 
 
 
 

 
 REF - naive_B 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 6.70 
 
 
 1 
 MALAT1 
 6.49 
 
 
 2 
 MT-CO1 
 5.80 
 
 
 3 
 EEF1A1 
 5.73 
 
 
 4 
 RPL41 
 5.68 
 
 
 5 
 RPS12 
 5.59 
 
 
 6 
 B2M 
 5.29 
 
 
 7 
 HLA-DRA 
 5.23 
 
 
 8 
 TMSB4X 
 5.01 
 
 
 9 
 RPS4X 
 4.96

B_Preplasma : 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 atypical_B 
 0.97 
 0.86 
 
 
 
 
 <!DOCTYPE html>
 
 
 
 
 Two Columns Layout 
 
 
 
 
 
 Overlap of DE genes 
 
 QUERY - B_Preplasma 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 LINC02137 
 9.15 
 0.000 
 4.64 
 
 
 1 
 MS4A1 
 7.29 
 0.000 
 109.97 
 
 
 2 
 SOX5 
 7.13 
 0.000 
 12.23 
 
 
 3 
 CD79A 
 7.10 
 0.000 
 108.58 
 
 
 4 
 GALNTL6 
 6.68 
 0.000 
 6.76 
 
 
 5 
 FCRL5 
 6.25 
 0.000 
 54.10 
 
 
 6 
 WNT16 
 6.06 
 0.000 
 5.06 
 
 
 7 
 EBI3 
 5.84 
 0.000 
 19.47 
 
 
 8 
 CD19 
 5.62 
 0.000 
 74.24 
 
 
 9 
 LINC01013 
 5.56 
 0.000 
 7.69 
 
 
 

 
 REF - atypical_B 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 MFAP5 
 9.28 
 0.000 
 4.56 
 
 
 1 
 LINC02137 
 9.07 
 0.000 
 5.85 
 
 
 2 
 MS4A1 
 8.03 
 0.000 
 105.12 
 
 
 3 
 SOX5 
 7.85 
 0.000 
 42.68 
 
 
 4 
 CD79A 
 7.56 
 0.000 
 103.02 
 
 
 5 
 FCRL5 
 7.19 
 0.000 
 71.35 
 
 
 6 
 FCRLA 
 6.43 
 0.000 
 92.33 
 
 
 7 
 CD19 
 6.42 
 0.000 
 90.74 
 
 
 8 
 BANK1 
 6.42 
 0.000 
 98.13 
 
 
 9 
 FGF2 
 6.35 
 0.004 
 3.10 
 
 
 
 
 
 
 Overlap of highly expressed genes 
 
 QUERY - B_Preplasma 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 6.69 
 
 
 1 
 MALAT1 
 6.65 
 
 
 2 
 EEF1A1 
 6.01 
 
 
 3 
 B2M 
 6.00 
 
 
 4 
 MT-CO1 
 5.79 
 
 
 5 
 RPS12 
 5.69 
 
 
 6 
 HLA-DRA 
 5.45 
 
 
 7 
 TMSB4X 
 5.43 
 
 
 8 
 RPL39 
 5.40 
 
 
 9 
 ACTB 
 5.10 
 
 
 

 
 REF - atypical_B 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 6.81 
 
 
 1 
 MALAT1 
 6.74 
 
 
 2 
 MT-CO1 
 5.79 
 
 
 3 
 B2M 
 5.75 
 
 
 4 
 RPL41 
 5.59 
 
 
 5 
 EEF1A1 
 5.53 
 
 
 6 
 RPS12 
 5.38 
 
 
 7 
 HLA-DRA 
 5.36 
 
 
 8 
 TMSB4X 
 5.11 
 
 
 9 
 ACTB 
 4.95

NKT : 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 CD4+_T_cyt 
 0.45 
 0.86 
 
 
 NK 
 0.45 
 1.04 
 
 
 
 
 <!DOCTYPE html>
 
 
 
 
 Two Columns Layout 
 
 
 
 
 
 Overlap of DE genes 
 
 QUERY - NKT 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 GNLY 
 7.01 
 0.000 
 204.94 
 
 
 1 
 NKG7 
 6.25 
 0.000 
 212.53 
 
 
 2 
 GZMB 
 5.72 
 0.000 
 202.30 
 
 
 3 
 CCL5 
 5.64 
 0.000 
 202.12 
 
 
 4 
 FGFBP2 
 5.46 
 0.000 
 192.57 
 
 
 5 
 GZMH 
 5.44 
 0.000 
 195.87 
 
 
 6 
 KLRC2 
 5.42 
 0.000 
 87.45 
 
 
 7 
 CST7 
 5.36 
 0.000 
 209.98 
 
 
 8 
 PRF1 
 5.30 
 0.000 
 196.24 
 
 
 9 
 TKTL1 
 5.29 
 0.000 
 49.67 
 
 
 

 
 REF - CD4+_T_cyt 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 GZMH 
 5.04 
 0.000 
 158.38 
 
 
 1 
 NKG7 
 4.85 
 0.000 
 122.13 
 
 
 2 
 CCL5 
 4.57 
 0.000 
 132.11 
 
 
 3 
 GNLY 
 4.40 
 0.000 
 109.07 
 
 
 4 
 FGFBP2 
 3.80 
 0.000 
 121.68 
 
 
 5 
 GZMA 
 3.67 
 0.000 
 124.27 
 
 
 6 
 CST7 
 3.48 
 0.000 
 112.40 
 
 
 7 
 PROK2 
 3.36 
 0.000 
 38.27 
 
 
 8 
 KIF19 
 3.25 
 0.000 
 19.78 
 
 
 9 
 IL5RA 
 3.10 
 0.000 
 23.63 
 
 
 

 
 REF - NK 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 GNLY 
 6.14 
 0.000 
 106.26 
 
 
 1 
 PPBP 
 6.11 
 0.000 
 122.72 
 
 
 2 
 NKG7 
 5.60 
 0.000 
 106.46 
 
 
 3 
 PF4 
 4.91 
 0.000 
 82.56 
 
 
 4 
 GZMB 
 4.91 
 0.000 
 104.70 
 
 
 5 
 GP1BB 
 4.90 
 0.000 
 76.23 
 
 
 6 
 TUBB1 
 4.81 
 0.000 
 72.33 
 
 
 7 
 CAVIN2 
 4.54 
 0.000 
 70.25 
 
 
 8 
 FGFBP2 
 4.49 
 0.000 
 97.71 
 
 
 9 
 SPARC 
 4.41 
 0.000 
 60.85 
 
 
 
 
 
 
 Overlap of highly expressed genes 
 
 QUERY - NKT 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 B2M 
 6.72 
 
 
 1 
 MALAT1 
 6.55 
 
 
 2 
 TMSB4X 
 5.93 
 
 
 3 
 ACTB 
 5.88 
 
 
 4 
 EEF1A1 
 5.66 
 
 
 5 
 NKG7 
 5.65 
 
 
 6 
 GNLY 
 5.50 
 
 
 7 
 MT-CO1 
 5.36 
 
 
 8 
 RPS12 
 5.29 
 
 
 9 
 HLA-C 
 5.08 
 
 
 

 
 REF - CD4+_T_cyt 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 MALAT1 
 6.71 
 
 
 1 
 B2M 
 6.49 
 
 
 2 
 MT-CO1 
 5.82 
 
 
 3 
 EEF1A1 
 5.72 
 
 
 4 
 TMSB4X 
 5.61 
 
 
 5 
 RPL41 
 5.60 
 
 
 6 
 RPS12 
 5.54 
 
 
 7 
 ACTB 
 5.49 
 
 
 8 
 HLA-C 
 5.31 
 
 
 9 
 RPS4X 
 4.92 
 
 
 

 
 REF - NK 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 MALAT1 
 6.48 
 
 
 1 
 B2M 
 6.32 
 
 
 2 
 MT-CO1 
 5.71 
 
 
 3 
 TMSB4X 
 5.63 
 
 
 4 
 ACTB 
 5.56 
 
 
 5 
 NKG7 
 5.33 
 
 
 6 
 HLA-C 
 5.30 
 
 
 7 
 EEF1A1 
 5.25 
 
 
 8 
 GNLY 
 5.24 
 
 
 9 
 RPL41 
 5.06

...

cDC_1 : 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 cDC1 
 0.99 
 0.69 
 
 
 
 
 <!DOCTYPE html>
 
 
 
 
 Two Columns Layout 
 
 
 
 
 
 Overlap of DE genes 
 
 QUERY - cDC_1 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 CLEC9A 
 12.68 
 0.000 
 36.08 
 
 
 1 
 IDO1 
 10.23 
 0.000 
 25.94 
 
 
 2 
 CLNK 
 9.15 
 0.000 
 24.91 
 
 
 3 
 DNASE1L3 
 8.13 
 0.000 
 23.11 
 
 
 4 
 CCND1 
 7.03 
 0.000 
 13.77 
 
 
 5 
 WDFY4 
 6.31 
 0.000 
 32.89 
 
 
 6 
 CST3 
 6.27 
 0.000 
 37.68 
 
 
 7 
 HLA-DQA1 
 6.18 
 0.000 
 36.56 
 
 
 8 
 EGLN3 
 6.08 
 0.000 
 12.77 
 
 
 9 
 SERPINF1 
 6.01 
 0.000 
 25.49 
 
 
 

 
 REF - cDC1 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 CLEC9A 
 13.33 
 0.000 
 44.65 
 
 
 1 
 IDO1 
 11.53 
 0.000 
 39.24 
 
 
 2 
 CLNK 
 9.14 
 0.000 
 40.27 
 
 
 3 
 DNASE1L3 
 8.97 
 0.000 
 37.65 
 
 
 4 
 CST3 
 7.23 
 0.000 
 44.73 
 
 
 5 
 CCND1 
 7.01 
 0.000 
 28.20 
 
 
 6 
 KCND3 
 6.91 
 0.000 
 7.41 
 
 
 7 
 WDFY4 
 6.85 
 0.000 
 43.85 
 
 
 8 
 CPVL 
 6.83 
 0.000 
 44.57 
 
 
 9 
 HLA-DRA 
 6.77 
 0.000 
 44.68 
 
 
 
 
 
 
 Overlap of highly expressed genes 
 
 QUERY - cDC_1 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 7.05 
 
 
 1 
 ACTB 
 6.51 
 
 
 2 
 TMSB4X 
 6.04 
 
 
 3 
 HLA-DRA 
 5.97 
 
 
 4 
 CST3 
 5.94 
 
 
 5 
 HLA-DPA1 
 5.67 
 
 
 6 
 HLA-DRB1 
 5.63 
 
 
 7 
 EEF1A1 
 5.59 
 
 
 8 
 B2M 
 5.59 
 
 
 9 
 MT-CO1 
 5.54 
 
 
 

 
 REF - cDC1 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 7.17 
 
 
 1 
 ACTB 
 6.29 
 
 
 2 
 CST3 
 5.95 
 
 
 3 
 HLA-DRA 
 5.91 
 
 
 4 
 TMSB4X 
 5.87 
 
 
 5 
 HLA-DPA1 
 5.63 
 
 
 6 
 MT-CO1 
 5.59 
 
 
 7 
 HLA-DRB1 
 5.55 
 
 
 8 
 HLA-DPB1 
 5.52 
 
 
 9 
 MALAT1 
 5.46

cDC_2 : 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 cDC2 
 0.63 
 0.68 
 
 
 cDC 
 0.50 
 0.69 
 
 
 
 
 <!DOCTYPE html>
 
 
 
 
 Two Columns Layout 
 
 
 
 
 
 Overlap of DE genes 
 
 QUERY - cDC_2 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 FCER1A 
 9.55 
 0.000 
 116.33 
 
 
 1 
 ENHO 
 8.52 
 0.000 
 59.58 
 
 
 2 
 CLEC10A 
 7.14 
 0.000 
 113.17 
 
 
 3 
 CD1C 
 7.04 
 0.000 
 102.00 
 
 
 4 
 HLA-DRA 
 5.59 
 0.000 
 135.98 
 
 
 5 
 CST3 
 5.54 
 0.000 
 136.79 
 
 
 6 
 MSLN 
 5.53 
 0.000 
 9.71 
 
 
 7 
 HLA-DQA1 
 5.33 
 0.000 
 125.19 
 
 
 8 
 PLD4 
 5.00 
 0.000 
 90.02 
 
 
 9 
 HLA-DQB1 
 4.98 
 0.000 
 128.00 
 
 
 

 
 REF - cDC2 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 FCER1A 
 9.49 
 0.000 
 151.35 
 
 
 1 
 CD207 
 8.92 
 0.000 
 8.08 
 
 
 2 
 ENHO 
 7.95 
 0.000 
 118.81 
 
 
 3 
 CLEC10A 
 7.71 
 0.000 
 160.58 
 
 
 4 
 APOC1 
 6.69 
 0.000 
 13.21 
 
 
 5 
 CD1C 
 6.49 
 0.000 
 138.88 
 
 
 6 
 CST3 
 6.28 
 0.000 
 190.42 
 
 
 7 
 IL1R2 
 6.24 
 0.000 
 51.28 
 
 
 8 
 MSLN 
 6.14 
 0.000 
 7.22 
 
 
 9 
 HLA-DRA 
 6.03 
 0.000 
 187.11 
 
 
 

 
 REF - cDC 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 FCER1A 
 7.65 
 0.000 
 21.73 
 
 
 1 
 CLEC9A 
 7.12 
 0.000 
 4.18 
 
 
 2 
 ENHO 
 6.97 
 0.000 
 19.59 
 
 
 3 
 IDO1 
 6.51 
 0.000 
 3.86 
 
 
 4 
 CLEC10A 
 6.51 
 0.000 
 20.45 
 
 
 5 
 CST3 
 6.48 
 0.000 
 27.96 
 
 
 6 
 HLA-DRA 
 6.46 
 0.000 
 28.20 
 
 
 7 
 CD1C 
 6.41 
 0.000 
 21.29 
 
 
 8 
 HLA-DQA1 
 6.25 
 0.000 
 27.74 
 
 
 9 
 APOC1 
 6.19 
 0.006 
 3.15 
 
 
 
 
 
 
 Overlap of highly expressed genes 
 
 QUERY - cDC_2 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 6.44 
 
 
 1 
 ACTB 
 6.32 
 
 
 2 
 EEF1A1 
 5.82 
 
 
 3 
 TMSB4X 
 5.71 
 
 
 4 
 HLA-DRA 
 5.64 
 
 
 5 
 MT-CO1 
 5.61 
 
 
 6 
 B2M 
 5.56 
 
 
 7 
 MALAT1 
 5.50 
 
 
 8 
 CST3 
 5.39 
 
 
 9 
 FTH1 
 5.28 
 
 
 

 
 REF - cDC2 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 6.34 
 
 
 1 
 ACTB 
 5.88 
 
 
 2 
 MT-CO1 
 5.69 
 
 
 3 
 MALAT1 
 5.61 
 
 
 4 
 TMSB4X 
 5.41 
 
 
 5 
 FTH1 
 5.39 
 
 
 6 
 EEF1A1 
 5.37 
 
 
 7 
 HLA-DRA 
 5.34 
 
 
 8 
 CST3 
 5.23 
 
 
 9 
 RPL41 
 5.15 
 
 
 

 
 REF - cDC 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 6.67 
 
 
 1 
 ACTB 
 5.99 
 
 
 2 
 HLA-DRA 
 5.70 
 
 
 3 
 MT-CO1 
 5.67 
 
 
 4 
 TMSB4X 
 5.53 
 
 
 5 
 FTH1 
 5.45 
 
 
 6 
 CST3 
 5.44 
 
 
 7 
 MALAT1 
 5.44 
 
 
 8 
 EEF1A1 
 5.36 
 
 
 9 
 HLA-DRB1 
 5.23

cM : 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 CD14+_Monocyte 
 0.99 
 0.57 
 
 
 
 
 <!DOCTYPE html>
 
 
 
 
 Two Columns Layout 
 
 
 
 
 
 Overlap of DE genes 
 
 QUERY - cM 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 S100A8 
 7.13 
 0.000 
 580.36 
 
 
 1 
 S100A12 
 7.09 
 0.000 
 521.94 
 
 
 2 
 CD14 
 7.01 
 0.000 
 515.14 
 
 
 3 
 LYZ 
 6.98 
 0.000 
 577.06 
 
 
 4 
 S100A9 
 6.92 
 0.000 
 579.12 
 
 
 5 
 VCAN 
 6.68 
 0.000 
 529.97 
 
 
 6 
 FCN1 
 6.56 
 0.000 
 544.67 
 
 
 7 
 CST3 
 6.23 
 0.000 
 522.72 
 
 
 8 
 HP 
 6.15 
 0.000 
 73.89 
 
 
 9 
 SERPINB10 
 6.12 
 0.000 
 10.67 
 
 
 

 
 REF - CD14+_Monocyte 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 S100A8 
 9.11 
 0.000 
 651.47 
 
 
 1 
 S100A9 
 8.63 
 0.000 
 651.79 
 
 
 2 
 LYZ 
 8.10 
 0.000 
 647.47 
 
 
 3 
 S100A12 
 7.65 
 0.000 
 568.84 
 
 
 4 
 VCAN 
 7.42 
 0.000 
 612.40 
 
 
 5 
 CD14 
 7.00 
 0.000 
 575.64 
 
 
 6 
 FCN1 
 6.93 
 0.000 
 621.91 
 
 
 7 
 CST3 
 6.59 
 0.000 
 592.51 
 
 
 8 
 CSF3R 
 6.50 
 0.000 
 562.77 
 
 
 9 
 IFI30 
 6.37 
 0.000 
 594.29 
 
 
 
 
 
 
 Overlap of highly expressed genes 
 
 QUERY - cM 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 S100A9 
 6.19 
 
 
 1 
 S100A8 
 6.19 
 
 
 2 
 FTL 
 6.00 
 
 
 3 
 ACTB 
 5.75 
 
 
 4 
 MT-CO1 
 5.53 
 
 
 5 
 MALAT1 
 5.53 
 
 
 6 
 TMSB4X 
 5.47 
 
 
 7 
 B2M 
 5.44 
 
 
 8 
 FTH1 
 5.39 
 
 
 9 
 LYZ 
 5.34 
 
 
 

 
 REF - CD14+_Monocyte 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 FTL 
 5.91 
 
 
 1 
 FTH1 
 5.68 
 
 
 2 
 MALAT1 
 5.66 
 
 
 3 
 S100A9 
 5.50 
 
 
 4 
 S100A8 
 5.45 
 
 
 5 
 ACTB 
 5.41 
 
 
 6 
 MT-CO1 
 5.38 
 
 
 7 
 LYZ 
 5.31 
 
 
 8 
 TMSB4X 
 5.22 
 
 
 9 
 FOS 
 5.19

ncM : 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 CD16+_Monocyte 
 1.00 
 0.54 
 
 
 
 
 <!DOCTYPE html>
 
 
 
 
 Two Columns Layout 
 
 
 
 
 
 Overlap of DE genes 
 
 QUERY - ncM 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 LYPD2 
 7.88 
 0.000 
 43.40 
 
 
 1 
 VMO1 
 7.47 
 0.000 
 45.97 
 
 
 2 
 CDKN1C 
 7.39 
 0.000 
 134.62 
 
 
 3 
 C1QB 
 6.86 
 0.000 
 54.81 
 
 
 4 
 C1QC 
 6.71 
 0.000 
 39.41 
 
 
 5 
 C1QA 
 6.60 
 0.000 
 95.41 
 
 
 6 
 CKB 
 6.35 
 0.000 
 65.29 
 
 
 7 
 MEG3 
 6.28 
 0.000 
 5.82 
 
 
 8 
 HES4 
 5.84 
 0.000 
 130.36 
 
 
 9 
 FCGR3A 
 5.68 
 0.000 
 225.09 
 
 
 

 
 REF - CD16+_Monocyte 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 LYPD2 
 8.47 
 0.000 
 113.91 
 
 
 1 
 CDKN1C 
 7.84 
 0.000 
 292.62 
 
 
 2 
 MEG3 
 7.75 
 0.000 
 12.31 
 
 
 3 
 VMO1 
 7.37 
 0.000 
 92.40 
 
 
 4 
 C1QA 
 7.00 
 0.000 
 96.00 
 
 
 5 
 C1QB 
 6.77 
 0.000 
 30.28 
 
 
 6 
 CKB 
 6.64 
 0.000 
 167.07 
 
 
 7 
 HES4 
 6.40 
 0.000 
 258.89 
 
 
 8 
 PELATON 
 6.35 
 0.000 
 320.56 
 
 
 9 
 LST1 
 6.11 
 0.000 
 340.88 
 
 
 
 
 
 
 Overlap of highly expressed genes 
 
 QUERY - ncM 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 FTL 
 6.40 
 
 
 1 
 FTH1 
 6.16 
 
 
 2 
 ACTB 
 6.09 
 
 
 3 
 B2M 
 5.90 
 
 
 4 
 TMSB4X 
 5.83 
 
 
 5 
 MALAT1 
 5.72 
 
 
 6 
 EEF1A1 
 5.50 
 
 
 7 
 MT-CO1 
 5.49 
 
 
 8 
 S100A4 
 5.24 
 
 
 9 
 CD74 
 4.91 
 
 
 

 
 REF - CD16+_Monocyte 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 FTL 
 6.22 
 
 
 1 
 FTH1 
 6.05 
 
 
 2 
 MALAT1 
 5.88 
 
 
 3 
 ACTB 
 5.86 
 
 
 4 
 MT-CO1 
 5.62 
 
 
 5 
 TMSB4X 
 5.58 
 
 
 6 
 B2M 
 5.43 
 
 
 7 
 EEF1A1 
 5.21 
 
 
 8 
 CD74 
 4.97 
 
 
 9 
 RPL41 
 4.95

pDC : 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 pDC 
 1.00 
 0.57 
 
 
 
 
 <!DOCTYPE html>
 
 
 
 
 Two Columns Layout 
 
 
 
 
 
 Overlap of DE genes 
 
 QUERY - pDC 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 SCT 
 13.70 
 0.000 
 56.81 
 
 
 1 
 LRRC26 
 13.27 
 0.000 
 53.40 
 
 
 2 
 SHD 
 12.33 
 0.000 
 34.56 
 
 
 3 
 CLEC4C 
 11.70 
 0.000 
 63.44 
 
 
 4 
 KRT5 
 11.69 
 0.000 
 12.72 
 
 
 5 
 LILRA4 
 11.51 
 0.000 
 74.36 
 
 
 6 
 PTPRS 
 9.57 
 0.000 
 36.39 
 
 
 7 
 TPM2 
 9.38 
 0.000 
 59.52 
 
 
 8 
 SERPINF1 
 9.35 
 0.000 
 70.86 
 
 
 9 
 PTCRA 
 8.96 
 0.000 
 48.57 
 
 
 

 
 REF - pDC 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 LRRC26 
 13.70 
 0.000 
 91.22 
 
 
 1 
 SCT 
 13.30 
 0.000 
 89.52 
 
 
 2 
 SHD 
 12.80 
 0.000 
 58.25 
 
 
 3 
 LILRA4 
 11.96 
 0.000 
 101.52 
 
 
 4 
 CLEC4C 
 11.83 
 0.000 
 97.11 
 
 
 5 
 KRT5 
 11.50 
 0.000 
 31.41 
 
 
 6 
 PTPRS 
 10.19 
 0.000 
 91.63 
 
 
 7 
 SERPINF1 
 9.87 
 0.000 
 101.49 
 
 
 8 
 LAMP5 
 9.82 
 0.000 
 76.40 
 
 
 9 
 DNASE1L3 
 9.40 
 0.000 
 79.05 
 
 
 
 
 
 
 Overlap of highly expressed genes 
 
 QUERY - pDC 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 6.54 
 
 
 1 
 MALAT1 
 6.36 
 
 
 2 
 B2M 
 6.10 
 
 
 3 
 EEF1A1 
 6.08 
 
 
 4 
 RPS12 
 5.74 
 
 
 5 
 ACTB 
 5.55 
 
 
 6 
 MT-CO1 
 5.49 
 
 
 7 
 FTH1 
 5.24 
 
 
 8 
 RPL39 
 5.17 
 
 
 9 
 TMSB4X 
 5.05 
 
 
 

 
 REF - pDC 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 6.54 
 
 
 1 
 MALAT1 
 6.45 
 
 
 2 
 B2M 
 5.65 
 
 
 3 
 EEF1A1 
 5.65 
 
 
 4 
 RPS12 
 5.46 
 
 
 5 
 MT-CO1 
 5.33 
 
 
 6 
 RPL41 
 5.31 
 
 
 7 
 ACTB 
 5.17 
 
 
 8 
 FTH1 
 5.08 
 
 
 9 
 RPS4X 
 4.91